# Train the glaucoma models on ACRIMA + REFUGE (Google Colab)

Not yet run (written without access to Colab). If a cell fails, read its error: most are a missing file or folder
and the message says which. Use **Runtime > Change runtime type > GPU** first.

What is trained on what:
- **ACRIMA**: healthy-only autoencoder (anomaly score), supervised baseline, and the classification test split.
- **REFUGE**: optic disc and cup segmentation, which gives the cup-to-disc ratio.

Screening aid only. Not a medical diagnosis.

In [ ]:
import torch, subprocess
print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "no GPU (training will be slow)")

## 1. Get the project
Upload `glaucoma-anomaly.zip` (the file you downloaded) when asked.

In [ ]:
from google.colab import files
up = files.upload()                      # choose glaucoma-anomaly.zip
import zipfile, os
zname = next(iter(up))
zipfile.ZipFile(zname).extractall("/content")
%cd /content/glaucoma-anomaly
!pip -q install numpy pandas scikit-learn opencv-python-headless pillow pyyaml matplotlib flask

## 2. Data
**ACRIMA** is open and is attempted automatically. **REFUGE** needs a free account, so you place it yourself.

In [ ]:
!python scripts/download_data.py

If ACRIMA did not download: get it from the figshare link printed above (or the Kaggle mirror), unzip it, and put
the images in `/content/glaucoma-anomaly/data/raw/acrima/`.

**REFUGE**: download it from https://refuge.grand-challenge.org (or the mirror in `docs/DATA.md`), put the images and
masks in your Google Drive, and set `REFUGE_DRIVE_FOLDER` below to that folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
REFUGE_DRIVE_FOLDER = "/content/drive/MyDrive/REFUGE"      # <-- change to where you put it
ACRIMA_DRIVE_FOLDER = None                                   # e.g. "/content/drive/MyDrive/ACRIMA" if you stored it there
import shutil, pathlib
for src, dst in [(REFUGE_DRIVE_FOLDER, "data/raw/refuge"), (ACRIMA_DRIVE_FOLDER, "data/raw/acrima")]:
    if src:
        assert pathlib.Path(src).exists(), f"Folder not found: {src}"
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("copied", src, "->", dst)

## 3. Check the data before training
Counts must be ACRIMA 705 / 396 / 309, and REFUGE masks must show cup/disc/background values.

In [ ]:
!python scripts/inspect_data.py

Read the output. If the REFUGE mask values are not `0, 128, 255`, or no image+mask pairs were found, stop and fix
`masks_from_refuge` in `src/glaucoma/cdr.py` (see `docs/DATA.md`). Do not train on data the code cannot read.

## 4. Smoke test (about 2 minutes), then the real run

In [ ]:
!python -m unittest discover -s tests 2>&1 | tail -5
!python scripts/run_pipeline.py --quick

In [ ]:
!rm -rf results/quick checkpoints          # remove smoke-test outputs
!python scripts/run_pipeline.py

## 5. Results
These are the numbers for your paper. They come from the test split only.

In [ ]:
print(open("results/tables/main_results.md").read())
import json; m = json.load(open("results/metrics.json"))
print("Segmentation Dice:", m.get("segmenter"))
print("Mean anomaly score on test (normal vs glaucoma):", m["autoencoder"]["test_mean_error"])

**Check before you trust them:** if the autoencoder AUC is near 0.5 it is not separating glaucoma from healthy
(see `docs/TROUBLESHOOTING.md`). Report that honestly. Also note the confidence intervals, which are wide on small
test sets.

## 6. Download the trained models and put them in your project

In [ ]:
!zip -qr trained_models.zip checkpoints results
from google.colab import files
files.download("trained_models.zip")

On your own computer: extract `trained_models.zip` into the project folder (so `checkpoints/` and `results/` sit next
to `backend/`), then:

```
python -m backend.app
cd frontend && cp .env.example .env && npm install && npm run dev
```

Open http://localhost:5173. `python scripts/doctor.py` should show the checkpoints as PASS.